In [4]:
import kagglehub
path = kagglehub.dataset_download("irkaal/foodcom-recipes-and-reviews")
print(path)

100%|██████████| 723M/723M [00:07<00:00, 94.9MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/irkaal/foodcom-recipes-and-reviews/versions/2


In [5]:
import os, pandas as pd
print(os.listdir(path))

df = pd.read_csv(path + "/recipes.csv")
print(df.shape)
print(*df.columns, sep="\n")

['reviews.parquet', 'recipes.csv', 'reviews.csv', 'recipes.parquet']
(522517, 28)
RecipeId
Name
AuthorId
AuthorName
CookTime
PrepTime
TotalTime
DatePublished
Description
Images
RecipeCategory
Keywords
RecipeIngredientQuantities
RecipeIngredientParts
AggregatedRating
ReviewCount
Calories
FatContent
SaturatedFatContent
CholesterolContent
SodiumContent
CarbohydrateContent
FiberContent
SugarContent
ProteinContent
RecipeServings
RecipeYield
RecipeInstructions


In [6]:
import re

def clean_r_vector(val):
    if pd.isna(val):
        return ""
    val = str(val)
    if val == "character(0)":
        return ""
    if val.startswith("c("):
        items = re.findall(r'"((?:[^"\\]|\\.)*)"', val)
        return ", ".join(i.replace('\\"', '"') for i in items)
    return val

# 1. Ambil kolom yang dipakai, buang duplikat & nama kosong
df = df[["RecipeId", "Name", "Description", "RecipeCategory",
         "Keywords", "RecipeIngredientParts", "RecipeInstructions",
         "AggregatedRating"]]
df = df.dropna(subset=["Name"]).drop_duplicates("RecipeId")

# 2. Sampel 100 ribu baris (tulis di laporan bahwa pakai sampel)
df = df.sample(100000, random_state=42).copy()

# 3. Bersihkan kolom list gaya R
for col in ["Keywords", "RecipeIngredientParts", "RecipeInstructions"]:
    df[col] = df[col].apply(clean_r_vector)
df["Description"] = df["Description"].fillna("")
df["RecipeCategory"] = df["RecipeCategory"].fillna("")

# 4. Rename sesuai CREATE TABLE
df.columns = ["recipe_id", "name", "description", "category",
              "keywords", "ingredients", "instructions", "rating"]

# 5. Cek hasil & simpan
print(df.shape)
print(df[["name", "keywords", "ingredients"]].head(3).to_string())
df.to_csv("recipes_clean.csv", index=False)

(100000, 8)
                                                          name                                                                                                         keywords                                                                                                                                                                      ingredients
360511                                    Picnic Cowboy Caviar  Beans, Tex Mex, Southwestern U.S., Potluck, Spring, Summer, No Cook, Refrigerator, < 4 Hours, Easy, Inexpensive  black beans, black-eyed peas, white corn, roma tomatoes, bell pepper, red onion, avocados, cilantro leaf, red wine vinegar, extra virgin olive oil, salt, pepper, cumin, garlic
392362  Turkey Schnitzel With Tots &amp; Apple-Cranberry Sauce                                                                                                  Meat, < 60 Mins                                                        potatoes, mcintosh apples, apple cider, dried sweet

In [7]:
import html
for col in ["name", "description", "keywords", "ingredients", "instructions"]:
    df[col] = df[col].apply(lambda x: html.unescape(x) if isinstance(x, str) else x)

df.to_csv("recipes_clean.csv", index=False)

from google.colab import files
files.download("recipes_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>